In [0]:
master_path = "/Volumes/logistics/bronze/volume/master_data/azure_sql_seed/"

carrier_master_path   = master_path + "carrier_master.csv"
route_master_path     = master_path + "route_master.csv"
warehouse_master_path = master_path + "warehouse_master.csv" 
seed_tables_sql_path  = master_path + "azure_sql_create_seed_tables.sql"  # Azure SQL (T-SQL) DDL for the 3 master tables

In [0]:
def read_master(path):
    return (spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(path))

carrier_df   = read_master(carrier_master_path)
route_df     = read_master(route_master_path)
warehouse_df = read_master(warehouse_master_path)


In [0]:
# T-SQL DDL for Azure SQL - read as text for reference (not runnable in Databricks SQL)
seed_sql_df = spark.read.text(seed_tables_sql_path, wholetext=True)
seed_sql = seed_sql_df.first()[0]
print(seed_sql)

In [0]:
server = "capstone100dbserver.database.windows.net"
database = "capstone2shipment"

jdbc_url = (
    f"jdbc:sqlserver://{server}:1433;"
    f"databaseName={database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
)
#
properties = {
    "user": dbutils.secrets.get("azure-sql-scope", "sql-user"),
    "password": dbutils.secrets.get("azure-sql-scope", "sql-pwd"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}   

In [0]:
import re
import time

# Split the T-SQL script into batches on GO (GO is a client-side separator, not T-SQL)
batches = [b.strip() for b in re.split(r"(?im)^\s*GO\s*;?\s*$", seed_sql) if b.strip()]
print(f"{len(batches)} batches found")

def run_tsql(statement, retries=3, wait_s=30):
    # Retry transient connection failures (e.g. Azure SQL resuming from auto-pause)
    for attempt in range(1, retries + 1):
        try:
            return _run_tsql(statement)
        except Exception as e:
            if "FAILED_JDBC.CONNECTION" not in str(e) or attempt == retries:
                raise
            print(f"  connection failed (attempt {attempt}/{retries}), retrying in {wait_s}s...")
            time.sleep(wait_s)

def _run_tsql(statement):
    # Spark JDBC runs sessionInitStatement on the opened connection before the query
    # -> used here to execute DDL. Script is idempotent (IF OBJECT_ID ... IS NULL), so re-runs are safe.
    (spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", properties["user"])
        .option("password", properties["password"])
        .option("driver", properties["driver"])
        .option("sessionInitStatement", statement)
        .option("query", "SELECT 1 AS ok")
        .load()
        .collect())

for i, batch in enumerate(batches, 1):
    print(f"Running batch {i}: {batch.splitlines()[0]}")
    run_tsql(batch)

print("Seed SQL executed")

In [0]:
verify_query = """
SELECT t.TABLE_SCHEMA, t.TABLE_NAME, COUNT(c.COLUMN_NAME) AS column_count
FROM INFORMATION_SCHEMA.TABLES t
JOIN INFORMATION_SCHEMA.COLUMNS c
  ON c.TABLE_SCHEMA = t.TABLE_SCHEMA AND c.TABLE_NAME = t.TABLE_NAME
WHERE t.TABLE_SCHEMA = 'dbo'
  AND t.TABLE_NAME IN ('carrier_master', 'warehouse_master', 'route_master')
GROUP BY t.TABLE_SCHEMA, t.TABLE_NAME
"""

azure_tables_df = spark.read.jdbc(url=jdbc_url, table=f"({verify_query}) AS v", properties=properties)
display(azure_tables_df)

In [0]:
carrier_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.carrier_master",
    properties=properties
)

display(carrier_df)

In [0]:
warehouse_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.warehouse_master",
    properties=properties
)

display(warehouse_df)

In [0]:
route_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.route_master",
    properties=properties
)

display(route_df)

In [0]:
from pyspark.sql import functions as F

bronze_schema = "logistics.bronze"

master_dfs = {
    "carrier_master":   carrier_df,
    "warehouse_master": warehouse_df,
    "route_master":     route_df,
}

for table, df in master_dfs.items():
    (df.withColumns({
            "_source_system":       F.lit("azure_sql"),
            "_source_table":        F.lit(f"{database}.dbo.{table}"),
            "_ingestion_timestamp": F.current_timestamp(),
        })
       .write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{bronze_schema}.{table}"))
    print(f"Created {bronze_schema}.{table}")

# Verify: source dataframe rows vs Delta table rows
display(spark.createDataFrame(
    [(t, df.count(), spark.table(f"{bronze_schema}.{t}").count()) for t, df in master_dfs.items()],
    "table string, source_rows long, delta_rows long"))